In [1]:
from pathlib import Path
import sys
import json
import csv
from collections import defaultdict

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

In [2]:
from src.adapter.GENEVA_adepter import GENEVAAdapter

from src.unified_format.event_schema import EventSchema

In [3]:
with open(ROOT/"data/raw/GENEVA/train.json", "r", encoding="utf-8") as file:
    data = json.loads(next(file))
# data.keys()

In [4]:
adapter = GENEVAAdapter()
sample = adapter.adapt(data)

print(sample.events)

[Event(event_type='Catastrophe', trigger=[Trigger(text='Disaster', span=[0, 1])], arguments=[Argument(role='Undesirable_event', mentions={'text': 'Disaster', 'span': [0, 1]})]), Event(event_type='Request', trigger=[Trigger(text='urged', span=[3, 4])], arguments=[Argument(role='Message', mentions={'text': 'Disaster readiness fund', 'span': [0, 3]})])]


In [5]:
mapping_by_event = defaultdict(list)

with open(
    ROOT / "data/raw/GENEVA/fn2geneva_mapping_annotations.tsv",
    "r",
    encoding="utf-8-sig",
    newline=""
) as file:
    reader = csv.DictReader(file, delimiter="\t")

    for row in reader:
        event_name = row.get("# Event Name", "").strip()
        if event_name:
            current_event = event_name
            continue
        else:
            is_argument_role = row.get("Is Argument Role?", "").strip()
            if is_argument_role == "1":
                role = row.get("Frame Element Name", "").strip()

                if role:
                    mapping_by_event[current_event].append(role)  

In [7]:
for event_type, arguments in mapping_by_event.items():
    event_schema = EventSchema(event_type=event_type, argument_roles=arguments)
    print(event_schema)
    break

EventSchema(event_type='Know', argument_roles=['Instrument', 'Evidence', 'Cognizer', 'Topic', 'Phenomenon', 'Means'])
